[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Federated_Learning_Privacy.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Federated Learning & Privacy

Train on data you never collect: FedAvg across simulated clients (with the non-IID failure mode demonstrated, not just mentioned), and differential privacy's ε explained by actually running the attack it prevents.

## 1. Pre-requisites

[Training Dynamics](./Training_Dynamics.ipynb), [Concentration](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) for the privacy-noise trade.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *FedAvg & the Non-IID Problem* (~40 min)
**Goal:** average client models instead of pooling data; watch heterogeneity bite, then patch it.
**Feeds into:** Session 2 (differential privacy).

---

## 2. The Averaged Classroom

💡 **Intuition.** Hospitals/phones can't ship their data; FedAvg ships the *model*: each client trains locally a few epochs, the server averages the weights, repeat. With IID clients this tracks centralized training closely. The demon is **non-IID data**: when each client sees only its own slice of the world, local training drags each model toward its slice (*client drift*), and the average of specialists is not a generalist. Fewer local steps (more communication) is the classic dial.

In [ ]:
# 4-class spiral-ish task; clients get IID or PATHOLOGICALLY split (one class each)

# YOUR CODE HERE


**What just happened.** Three FedAvg runs on the same task and the same model, differing only in **how the data was split** and **how long clients train between syncs**. The IID curve climbs smoothly. The non-IID curve with 5 local epochs is visibly worse — and the non-IID curve with 1 local epoch recovers much of the gap.

**Nothing about the algorithm changed between curves two and three.** Same clients, same data split, same model, same learning rate. **Only the number of local steps between averages.** That isolation is what makes this an experiment rather than an illustration: the degradation is attributable to one variable.

**The mechanism is client drift, and it is worth deriving rather than naming.** In the non-IID split each client holds **exactly one class**. A client with only class 2 has a locally optimal model that predicts class 2 for every input — and five local epochs drag it a long way toward that degenerate solution. Average four such specialists and you do not get a generalist. **Averaging weights is not averaging functions**, and in a non-convex landscape the midpoint of four distant parameter vectors has no reason to be good at all.

**Which explains why fewer local steps helps.** Averaging is safe while the client models stay close enough that the loss surface between them is approximately quadratic. One local epoch keeps them close; five lets them wander into regions where interpolation is meaningless. **Local epochs are a dial on how far the models are allowed to drift before being forced back together.**

**But the fix is not free, and the cost is the one that matters in practice.** Going from 5 local epochs to 1 means **5× as many communication rounds** for the same local computation. In cross-device federated learning — millions of phones on metered, intermittent connections — communication *is* the bottleneck, and multiplying it by five can be prohibitive. **The dial trades the expensive resource for the cheap one, and which is which depends on the deployment**, not on the algorithm.

**That trade is why the algorithmic alternatives exist.** FedProx adds a proximal penalty keeping each local model near the global one; SCAFFOLD estimates each client's drift direction and corrects for it explicitly. Both attack drift *without* paying the full communication bill. **"Communicate more" is the honest baseline and frequently unaffordable**, which is what the literature is responding to.

**Finally, note three ways this simulation is easier than reality.** All four clients participate in **every** round — real systems sample a small fraction of currently-available devices. All clients hold **equal** amounts of data — real distributions are wildly imbalanced, and weighting the average by client size is essential. And no client drops out mid-round. **Each idealisation makes federated learning harder than this picture**, so read the curves as demonstrating a mechanism, not as forecasting a deployment.

**One thing federated learning has *not* achieved here, which Session 2 addresses.** The data never moved — and that is not the same as privacy. Model updates leak information about the data that produced them; gradient-inversion attacks can reconstruct training images from a single update. **Not shipping data is a good start and not a guarantee.**

---
### 🕐 Session 2 of 2 — *Differential Privacy, by Attack* (~40 min)
**Goal:** run a membership-style attack on released statistics; watch calibrated noise kill it — that's ε.
**Builds on:** Session 1.

---

## 3. What ε Actually Buys

💡 **Intuition.** Even *aggregates* leak: with an average over $n$ people and an attacker who knows everyone else, the release reveals the last person exactly. **Differential privacy** injects noise calibrated to the *sensitivity* (how much one person can move the output): the Laplace mechanism releases $f(D) + \mathrm{Lap}(\Delta f/\varepsilon)$, guaranteeing that any released value is at most $e^\varepsilon$ times more likely under your presence than your absence — your inclusion is *statistically deniable*. We don't recite that; we run the attack.

In [ ]:
# the difference attack: attacker knows all salaries but Alice's; the mean is released
# exact release: Alice recovered to the cent
# DP release: the attack's guess distribution vs epsilon

# YOUR CODE HERE


**What just happened.** With no privacy, the attacker recovered Alice's salary as **75.5560** against a truth of **75.5560** — exact to four decimal places, from a released *average*. Then noise was added and the attack degraded:

| ε | attacker's guess | verdict |
|---|---|---|
| ∞ (none) | 75.5560 (exact) | total compromise |
| 10 | 75.5 ± 23.4 | **still damaging** |
| 1 | 72.5 ± 224.9 | useless |
| 0.1 | 82.0 ± 2269.5 | useless |

**The first row is the point of the session, and it should be uncomfortable.** Nothing was released except a mean over 100 people. The attacker knew the other 99 and computed $100\bar{x} - \sum_{i\ne\text{Alice}}x_i$. **Aggregation is not anonymisation** — two lines of arithmetic, and the aggregate reveals an individual exactly.

**Check the noisy rows against theory rather than eyeballing them.** The Laplace scale is $b = \Delta f/\varepsilon = 1.6/\varepsilon$, its standard deviation is $b\sqrt2$, and the attacker multiplies the released mean by $n = 100$. So the predicted spread is $100 \times 1.6\sqrt2/\varepsilon = 226/\varepsilon$: **22.6, 226, 2263** at the three ε values. Measured: **23.4, 224.9, 2269.5**. Agreement to three significant figures — this is a verification, not an illustration.

**Now the row that deserves the most attention: ε = 10 is not private.** The guess is centred on the truth and has a spread of ±23k. **A $\pm$23k window on someone's salary is a serious disclosure** — it separates a junior engineer from a director. Yet $\varepsilon = 10$ satisfies the formal definition of differential privacy. **A guarantee with a badly chosen parameter is a bad guarantee**, and this is exactly why practitioners argue about ε values rather than treating DP as a binary property.

**Read the guarantee correctly while the numbers are on screen.** DP does not say "the attacker learns nothing". It says any output is at most $e^\varepsilon$ times more likely with you in the dataset than without — **your participation becomes statistically deniable**. At $\varepsilon = 10$ that ratio is $e^{10} \approx 22{,}000$, which is essentially no constraint. At $\varepsilon = 1$ it is 2.7, and the attack collapses.

**And note what makes DP unusual as a security property.** The guarantee holds against **any** attacker with **any** side information, including an attacker who knows all 99 other salaries — which is exactly the adversary that broke the un-noised release. **It is a property of the mechanism, not an assumption about the adversary**, and that universal quantifier is why DP displaced ad-hoc anonymisation schemes.

**One assumption in the code is doing quiet work and is where real deployments break.** `sensitivity=160/n_people` assumes every salary lies in $[0, 160]$. **If one person earns 500, the true sensitivity is larger, the noise is too small, and the ε guarantee is simply false.** Real systems must clip values to an assumed range first, accepting bias in exchange for a bound that holds. The guarantee is only ever as good as the sensitivity bound it was computed from.

**Finally, the mechanism connects straight back to Session 1.** DP-SGD in federated learning is this same construction: **clip** each client's update to bound its sensitivity, then add calibrated noise to the aggregate. Averaging supplies the statistic; noise supplies the deniability. And the clipping is what makes the noisy sum analysable at all — the same bounded-summand hypothesis that [Hoeffding](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) needs.

In [ ]:
# and the price: utility of the released statistic vs ε (the privacy-utility frontier)

# YOUR CODE HERE


**What just happened.** The privacy–utility frontier, and on log–log axes it is a **straight line of slope $-1$**. The standard deviation of the released mean is $\Delta f \sqrt2/\varepsilon = 2.26/\varepsilon$ — so tenfold stronger privacy costs exactly tenfold more error, at every point on the curve.

**The slope is the content, not the intercept.** A straight line on log–log means a power law, and slope $-1$ means **error $\propto 1/\varepsilon$**. This is not an empirical regularity that better engineering might improve — it falls directly out of the Laplace mechanism, where the noise scale *is* $\Delta f/\varepsilon$ by construction. **Privacy and accuracy sit on a hyperbola, and no cleverness moves you off it.**

**Read the two ends against the attack results to price the trade concretely.** At $\varepsilon = 10$ the released mean is accurate to about $\pm0.23$k — excellent utility — and the attacker still pins Alice to $\pm23$k. At $\varepsilon = 0.1$ the attack is hopeless and the released mean carries $\pm23$k of noise, which for a statistic about 100 people is worthless. **Both ends are unusable for opposite reasons**, which is precisely why choosing ε is a policy decision rather than a technical one.

**There is one genuine escape from the frontier, and it is not on this plot.** Sensitivity is $\Delta f = R/n$, so **increasing $n$ shifts the entire curve down**. Ten thousand people instead of a hundred means 100× less noise at the same ε. That is the whole reason DP works well for population-level statistics — census tabulations, aggregate telemetry — and fails for small subgroups: **the rarer the group, the higher the privacy price per person**, which is a real equity problem in deployed DP systems, not a technicality.

**Note also what the plot does not show: composition.** Every query against the same dataset spends more budget. Answer $k$ questions at $\varepsilon$ each and the total is $k\varepsilon$ under basic composition (better under advanced composition, but still growing). **A privacy budget is a finite resource being consumed**, so a system answering thousands of queries must divide a fixed ε among them — and the per-query noise on this plot is the *best* case of a single release.

**Which is exactly how DP-SGD spends its budget.** Every training step is a query against the dataset, so a run of thousands of steps must allocate ε across all of them. That is why DP-trained models are noticeably worse than their non-private counterparts, and why the accounting machinery (moments accountant, Rényi DP) exists — to prove the total spend is smaller than naive composition suggests.

**Close by putting the two sessions together, since that is the toolkit.** DP-SGD **clips** each client's update to bound sensitivity, then adds this same calibrated noise to the aggregate. Session 1 supplies the averaging; Session 2 supplies the deniability; and this plot is the bill. **Nothing here is free, and the value of the workshop is that every cost was measured rather than asserted.**

## 4. Conclusion

FedAvg trades pooling for averaging (and pays a measured price under heterogeneity, refundable via communication); DP makes single-person influence deniable, with the attack's collapse — and the utility bill — both measured. Together they're the toolkit for learning from data nobody may hand you.

---
## Where next

- [Distributed Training II](../Intro_GPU/Distributed_Training_2.ipynb) — the same averaging, for speed instead of privacy.
- [Concentration](../Intro_Math/Concentration/Concentration_Inequalities.ipynb) — why clipped sums concentrate, which is what makes DP-SGD analyzable.